# 02a - Train DL Methods

Run after `01_preprocessing.ipynb` and before `02_run_methods.ipynb`.

1. trains one DL method on one or more folds,
2. writes checkpoints to `OUTPUT_ROOT`.

Classical methods (1a, 2, 3c, 4) skip this notebook entirely.

In [ ]:
import os, pathlib, sys

root = pathlib.Path.cwd()
while not (root / "requirements.txt").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.paths import resolve_roots
DATA_ROOT, OUTPUT_ROOT = resolve_roots()

print('Repo root  :', root)
print('DATA_ROOT  :', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
from src.s1_data.dataset import OCTDataset
from src.s1_data.splits import load_folds
from src.s3_methods.m5_deep_learning.training import CanvasDataset, train

# Swap this import for the DL method being trained, e.g.:
# from src.s3_methods.m5_deep_learning.d_unet import UNet

# DL model classes are currently NotImplementedError stubs, so this notebook cannot run end to end yet.
METHOD = 'unet'
FOLDS = [0]  # Stage 0 uses one fold; full sweep sets range(5)
SEED = 42

In [ ]:
folds = load_folds(DATA_ROOT / 'processed/folds.json')
histories = {}

for FOLD in FOLDS:
    train_ids, val_ids = folds[FOLD]
    train_set = CanvasDataset(
        [
            OCTDataset(DATA_ROOT / 'processed/duke_dme_denoised', patient_ids=train_ids),
            OCTDataset(DATA_ROOT / 'processed/hc_ms_denoised', patient_ids=train_ids),
        ],
        augment_samples=True,
        seed=SEED,
    )
    val_set = CanvasDataset(
        [
            OCTDataset(DATA_ROOT / 'processed/duke_dme_denoised', patient_ids=val_ids),
            OCTDataset(DATA_ROOT / 'processed/hc_ms_denoised', patient_ids=val_ids),
        ]
    )
    print(f'Fold {FOLD}: train={len(train_set)} scans, val={len(val_set)} scans')

    # Uncomment once the method's model class is implemented. Build a fresh
    # model per fold, never reuse one across folds.
    # model = UNet()
    # histories[FOLD] = train(
    #     model,
    #     train_set,
    #     val_set,
    #     seed=SEED,
    #     checkpoint_path=OUTPUT_ROOT / f'{METHOD}_fold{FOLD}_seed{SEED}.pt',
    # )
    # print(f'  best val_loss={min(s["val_loss"] for s in histories[FOLD]):.4f}')

In [ ]:
import matplotlib.pyplot as plt

# Plot train and val loss per epoch across folds
# for fold, history in histories.items():
#     epochs = [step['epoch'] for step in history]
#     train_loss = [step['train_loss'] for step in history]
#     val_loss = [step['val_loss'] for step in history]
#     plt.plot(epochs, train_loss, label=f'Fold {fold} Train')
#     plt.plot(epochs, val_loss, label=f'Fold {fold} Val')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.show()